# 06 — Ablasi: degradasi modalitas dan missing modality

Tahap 7. **Belum pernah dijalankan** di mesin penulis kode. Jalankan di mesin
compute.

Bagian ini **bukan** bagian dari paper. Inilah yang membedakan proyek dari sekadar
replikasi.

Penulis mengakui di bagian limitasi bahwa Tobacco3482 seluruhnya berorientasi rapi
dan discan profesional, sehingga tidak merepresentasikan kondisi nyata — tapi mereka
tidak pernah mengujinya. Ablasi 1 dan 3 adalah pengujian empiris atas limitasi yang
penulis sebut lalu tinggalkan.

## Tidak ada training di notebook ini

Ablasi 1-3 adalah perturbasi pada **inferensi**, jadi seluruhnya memakai checkpoint
Tahap 5-6 apa adanya. Ablasi 4 (fusion bergerbang / cross-attention) butuh training
baru dan **dilewati** — lihat catatan di akhir.

Satu seed saja (42) di seluruh tahap ini, dicatat sebagai keterbatasan.

## Caveat yang wajib masuk laporan

Kita mendegradasi citra sementara teksnya tetap berasal dari OCR atas citra **bersih**
aslinya. Di sistem nyata, scan yang blur menghasilkan OCR yang lebih buruk juga, jadi
kedua cabang akan memburuk bersamaan.

Artinya ablasi ini mengukur ketahanan **arsitektur**, bukan ketahanan sistem ujung ke
ujung. Itu tetap pertanyaan yang sah — "apakah cabang teks mengompensasi ketika
cabang citra rusak" — tapi jangan dibaca sebagai "sistem ini tahan terhadap dokumen
buruk". Lihat `vault/20-metode/taksonomi-multimodal.md`.

**Prasyarat:** `01`–`05` sudah dijalankan, dan checkpoint ada di `models/`.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from src import ablation as A
from src import datasets as DS
from src import splits as S
from src import train as T
from src.config import CONFIG, set_seed
from src.models import CNN1D, FusionModel, ImageBranch
from src.utils import save_fig

SEED = 42
set_seed(SEED)

manifest = pd.read_csv(CONFIG["data_processed"] / "manifest.csv")
labels = manifest["kelas"].tolist()
y = DS.label_indices(labels)
split = S.load_split(SEED, labels)
device = T.pick_device()
seq_path = CONFIG["data_processed"] / "sequences_500x300_fp16.npy"
paths = manifest["path_citra"].tolist()
print(f"seed {SEED} | test {len(split['test'])} dokumen | device {device}")

## 0. Muat checkpoint

Tiga model dari Tahap 5-6, dalam mode eval. Tidak ada optimizer, tidak ada
`backward()`.

In [ ]:
def muat(model, nama):
    ckpt = torch.load(CONFIG["models"] / nama, map_location="cpu", weights_only=True)
    model.load_state_dict(ckpt["state_dict"])
    print(f"{nama}: epoch {ckpt['epoch']}, val_oa {ckpt['val_oa']:.4f}")
    return model.to(device).eval()

m_image = muat(ImageBranch(pretrained=False), f"IMAGE_seed{SEED}.pt")
m_text = muat(CNN1D(), f"CNN1D_seed{SEED}.pt")
m_fusion = muat(FusionModel(strategy="concat", pretrained_image=False),
                f"FUSION-concat_seed{SEED}.pt")

In [ ]:
def eval_citra(transform):
    ds = DS.ImageDataset(paths, y, split["test"], transform)
    ld = torch.utils.data.DataLoader(ds, batch_size=CONFIG["batch_size"],
                                     num_workers=4)
    return T.evaluate(m_image, ld, device)

def buat_pasangan(transform, text_fraction=0.0):
    img = DS.ImageDataset(paths, y, split["test"], transform)
    txt = DS.SequenceDataset(seq_path, y, split["test"])
    pair = DS.PairDataset(img, txt)
    if text_fraction > 0:
        pair = A.DegradedPairDataset(pair, text_fraction, seed=SEED)
    return pair

def eval_fusion(transform, text_fraction=0.0, zero=None):
    ld = torch.utils.data.DataLoader(buat_pasangan(transform, text_fraction),
                                     batch_size=CONFIG["batch_size"], num_workers=4)
    if zero is None:
        return T.evaluate(m_fusion, ld, device)
    # blanking one modality needs to happen per batch, so evaluate by hand
    trues, preds = [], []
    with torch.no_grad():
        for (img, txt), yb in ld:
            batch = A.zero_modality((img.to(device), txt.to(device)), zero)
            preds.append(m_fusion(batch).argmax(1).cpu().numpy())
            trues.append(yb.numpy())
    yt, yp = np.concatenate(trues), np.concatenate(preds)
    from sklearn.metrics import f1_score
    return {"oa": float((yt == yp).mean()),
            "macro_f1": float(f1_score(yt, yp, average="macro")),
            "y_true": yt, "y_pred": yp}

bersih_img = eval_citra(A.degraded_transform("none"))
bersih_fus = eval_fusion(A.degraded_transform("none"))
print(f"\nbaseline bersih — IMAGE {bersih_img['oa']:.4f} | "
      f"FUSION {bersih_fus['oa']:.4f}")

## Ablasi 1 — degradasi citra bertahap

Empat jenis degradasi, masing-masing beberapa tingkat. Level pertama tiap jenis
adalah identitas, jadi titik pertama tiap kurva adalah baseline yang tidak disentuh —
itu sekaligus pemeriksaan bahwa pipeline-nya benar.

Rotasi diisi putih, bukan hitam: dokumen adalah tulisan gelap di latar putih, jadi
sudut hitam akan menyuntikkan sinyal artifisial besar dan ablasi berubah jadi
"apakah CNN bisa melihat segitiga hitam".

In [ ]:
baris = []
for kind, levels in A.IMAGE_LEVELS.items():
    for lv in levels:
        tf = A.degraded_transform(kind, lv, seed=SEED)
        oa_i = eval_citra(tf)["oa"]
        oa_f = eval_fusion(tf)["oa"]
        baris.append({"jenis": kind, "level": lv,
                      "oa_image": round(oa_i, 4), "oa_fusion": round(oa_f, 4),
                      "selisih": round(oa_f - oa_i, 4)})
        print(f"{kind:7s} level {lv:<5} IMAGE {oa_i:.4f}  FUSION {oa_f:.4f}  "
              f"({oa_f - oa_i:+.4f})")

df1 = pd.DataFrame(baris)
df1.to_csv(CONFIG["root"] / "reports" / "ablasi1_degradasi_citra.csv", index=False)
display(df1)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 8))
for ax, kind in zip(axes.flat, A.IMAGE_LEVELS):
    sub = df1[df1["jenis"] == kind]
    x = sub["level"].astype(float)
    ax.plot(x, sub["oa_image"], "o-", label="IMAGE", color="#4C78A8")
    ax.plot(x, sub["oa_fusion"], "s-", label="FUSION", color="#E45756")
    ax.set_title(kind)
    ax.set_xlabel("tingkat degradasi")
    ax.set_ylabel("overall accuracy")
    ax.set_ylim(0, 1)
    ax.grid(alpha=0.3)
    ax.legend()
    if kind == "jpeg":
        ax.invert_xaxis()   # quality turun = makin buruk
fig.suptitle("Akurasi vs degradasi citra — apakah cabang teks mengompensasi?")
fig.tight_layout()
save_fig(fig, "07_ablasi_degradasi_citra")

In [ ]:
# Pertanyaan intinya: apakah JARAK FUSION - IMAGE melebar saat citra makin rusak?
# Kalau melebar, cabang teks memang mengompensasi. Kalau datar, fusion hanya
# mewarisi ketahanan cabang citra.
print("Selisih FUSION - IMAGE, level teringan vs terberat per jenis:\n")
for kind, levels in A.IMAGE_LEVELS.items():
    sub = df1[df1["jenis"] == kind]
    awal, akhir = sub.iloc[0]["selisih"], sub.iloc[-1]["selisih"]
    arah = "MELEBAR" if akhir > awal + 0.01 else (
           "menyempit" if akhir < awal - 0.01 else "datar")
    print(f"{kind:7s} {awal:+.4f} -> {akhir:+.4f}   {arah}")
print("\nMELEBAR di sebagian besar jenis = bukti cabang teks mengompensasi.")
print("Datar atau menyempit = #temuan/negatif, dan lebih menarik: berarti fusion "
      "tidak menambah ketahanan, hanya akurasi pada data bersih.")

## Ablasi 2 — missing modality saat inferensi

Nol-kan salah satu masukan, lalu ukur seberapa jauh fusion kolaps.

Untuk teks, nol **persis** seperti padding, jadi sekuens yang dinolkan adalah
"dokumen kosong" yang benar-benar in-distribution — dataset kita memang sudah memuat
dokumen yang OCR-nya tidak menghasilkan apa pun.

Untuk citra tidak sebersih itu: tensornya sudah ternormalisasi, jadi nol berarti
**citra rata-rata ImageNet**, bukan hitam. Itu pilihan yang paling tidak
mengagetkan, tapi tetap pilihan, dan harus disebut di laporan.

In [ ]:
tf_bersih = A.degraded_transform("none")
baris2 = [
    {"kondisi": "keduanya utuh", "oa": bersih_fus["oa"],
     "macro_f1": bersih_fus["macro_f1"]},
]
for zero, nama in [("text", "teks dinolkan"), ("image", "citra dinolkan")]:
    ev = eval_fusion(tf_bersih, zero=zero)
    baris2.append({"kondisi": nama, "oa": round(ev["oa"], 4),
                   "macro_f1": round(ev["macro_f1"], 4)})

df2 = pd.DataFrame(baris2)
df2["turun_dari_utuh"] = (df2["oa"] - bersih_fus["oa"]).round(4)
display(df2)

oa_text_only = df2[df2["kondisi"] == "citra dinolkan"]["oa"].iloc[0]
oa_img_only = df2[df2["kondisi"] == "teks dinolkan"]["oa"].iloc[0]
print(f"\nFUSION dengan hanya teks : {oa_text_only:.4f}  "
      f"(baseline TEXT standalone sebagai pembanding)")
print(f"FUSION dengan hanya citra: {oa_img_only:.4f}  "
      f"(baseline IMAGE standalone {bersih_img['oa']:.4f})")
print("\nKalau 'teks dinolkan' hampir tidak menurunkan apa pun, cabang teks "
      "praktis tidak terpakai — itu risiko yang dicatat di 05-fusion-concat.")
print("Kalau 'citra dinolkan' jatuh ke sekitar level TEXT standalone, fusion "
      "memakai kedua modalitas dengan wajar.")

## Ablasi 3 — degradasi teks

Hapus sebagian kata secara acak. Kata dihapus lalu sekuens dirapatkan kembali dan
di-pad di ujung, **bukan** dinolkan di tempat: saat training, nol hanya pernah muncul
sebagai padding di ekor, jadi melubangi tengah sekuens itu out-of-distribution dengan
cara yang mencampur "teks lebih sedikit" dengan "masukan yang belum pernah dilihat
model".

`fraction = 1.0` setara ablasi 2 untuk sisi teks, dan dua angka itu harus cocok —
pemeriksaan silang yang bagus.

In [ ]:
baris3 = []
for frac in A.TEXT_LEVELS["drop_words"]:
    ev = eval_fusion(tf_bersih, text_fraction=frac)
    baris3.append({"fraksi_dihapus": frac, "oa": round(ev["oa"], 4),
                   "macro_f1": round(ev["macro_f1"], 4)})
    print(f"hapus {frac:>4.0%} kata -> FUSION OA {ev['oa']:.4f}")

df3 = pd.DataFrame(baris3)
df3["turun_dari_utuh"] = (df3["oa"] - bersih_fus["oa"]).round(4)
display(df3)

silang = abs(df3[df3["fraksi_dihapus"] == 1.0]["oa"].iloc[0] - oa_img_only)
print(f"\nPemeriksaan silang: hapus 100% kata vs teks dinolkan, selisih {silang:.4f}")
assert silang < 0.02, ("hapus semua kata dan nol-kan teks seharusnya setara; "
                       "selisih besar menandakan bug di salah satunya")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(df3["fraksi_dihapus"], df3["oa"], "o-", color="#E45756", label="FUSION")
ax.axhline(bersih_img["oa"], ls="--", color="#4C78A8",
           label=f"IMAGE standalone ({bersih_img['oa']:.3f})")
ax.set_xlabel("fraksi kata dihapus")
ax.set_ylabel("overall accuracy")
ax.set_title("Degradasi teks — sampai mana fusion bertahan?")
ax.grid(alpha=0.3)
ax.legend()
fig.tight_layout()
save_fig(fig, "07_ablasi_degradasi_teks")

print("Kalau kurva berhenti di atas garis IMAGE standalone bahkan saat teks habis, "
      "fusion masih mendapat sesuatu dari arsitekturnya sendiri.")
print("Kalau jatuh di BAWAH garis itu, fusion lebih buruk daripada memakai citra "
      "saja ketika teks hilang — kerapuhan yang layak dilaporkan.")

### Injeksi typo — butuh model FastText

`A.inject_typos` merusak karakter di dalam token, jadi token itu harus di-embed
ulang. Fitur yang sudah dihitung tidak cukup; `cc.en.300.bin` harus dimuat lagi
(~15 GB RAM).

Sel di bawah dilewati kalau modelnya tidak ada. Kalau dilewati, **catat sebagai
bagian ablasi 3 yang tidak dikerjakan**, jangan diam-diam dihilangkan dari laporan.

In [ ]:
ft_path = CONFIG["models"] / CONFIG["fasttext_model"]
if not ft_path.exists():
    print(f"{ft_path.name} tidak ada — injeksi typo DILEWATI.")
    print("Catat di 09-ablasi-degradasi-teks.md sebagai bagian yang tidak dikerjakan.")
    df3b = None
else:
    from src import text_features as TF

    tokenize = TF.spacy_tokenizer()
    embed = TF.fasttext_embedder(ft_path)
    teks_test = [Path(manifest["path_teks"][i]).read_text(encoding="utf-8",
                                                          errors="replace")
                 for i in split["test"]]
    tok_test = TF.tokenize_corpus(teks_test, tokenize)

    baris3b = []
    for frac in A.TEXT_LEVELS["typos"]:
        rusak = [A.inject_typos(t, frac, seed=SEED + i)
                 for i, t in enumerate(tok_test)]
        tmp = CONFIG["data_processed"] / f"tmp_typo_{frac}.npy"
        TF.build_sequences(rusak, embed, tmp)
        ds_txt = DS.SequenceDataset(tmp, y[split["test"]], range(len(rusak)))
        ds_img = DS.ImageDataset(paths, y, split["test"], tf_bersih)
        ld = torch.utils.data.DataLoader(DS.PairDataset(ds_img, ds_txt),
                                         batch_size=CONFIG["batch_size"])
        ev = T.evaluate(m_fusion, ld, device)
        baris3b.append({"fraksi_typo": frac, "oa": round(ev["oa"], 4),
                        "macro_f1": round(ev["macro_f1"], 4)})
        print(f"typo {frac:>4.0%} -> FUSION OA {ev['oa']:.4f}")
        tmp.unlink()

    df3b = pd.DataFrame(baris3b)
    display(df3b)
    print("\nBandingkan dengan penghapusan kata pada fraksi yang sama: kalau typo "
          "jauh lebih ringan dampaknya, itu bukti langsung bahwa subword FastText "
          "bekerja — inti argumen paper, diuji pada model yang sudah terlatih.")
    del embed

## Ablasi 4 — dilewati

Fusion bergerbang atau cross-attention sederhana sebagai pembanding concat.

**Dilewati dengan sengaja.** Rencana menandainya opsional, dan ini satu-satunya
ablasi yang butuh training dari awal — dua arsitektur baru x 200 epoch, di atas
anggaran compute yang sudah dipakai Tahap 5-6.

Tambahkan kalau: hasil ablasi 1-3 menunjukkan concat gagal memanfaatkan teks (jarak
FUSION - IMAGE datar, atau menolkan teks tidak menurunkan apa pun). Dalam kasus itu
gerbang eksplisit adalah hipotesis berikutnya yang paling jelas, dan pertanyaannya
jadi layak biaya training.

## Ringkasan untuk catatan eksperimen

Tiga catatan di `vault/30-eksperimen/`: 07 (degradasi citra), 08 (missing modality),
09 (degradasi teks).

In [ ]:
print("===== ABLASI 1 — degradasi citra =====")
for kind, levels in A.IMAGE_LEVELS.items():
    sub = df1[df1["jenis"] == kind]
    print(f"{kind}: OA IMAGE {sub.iloc[0]['oa_image']:.4f} -> "
          f"{sub.iloc[-1]['oa_image']:.4f} | FUSION "
          f"{sub.iloc[0]['oa_fusion']:.4f} -> {sub.iloc[-1]['oa_fusion']:.4f} | "
          f"selisih {sub.iloc[0]['selisih']:+.4f} -> {sub.iloc[-1]['selisih']:+.4f}")

print("\n===== ABLASI 2 — missing modality =====")
for _, r in df2.iterrows():
    print(f"{r['kondisi']:16s} oa:: {r['oa']:.4f}  macro_f1:: {r['macro_f1']:.4f}  "
          f"(turun {r['turun_dari_utuh']:+.4f})")

print("\n===== ABLASI 3 — degradasi teks =====")
for _, r in df3.iterrows():
    print(f"hapus {r['fraksi_dihapus']:>4.0%}  oa:: {r['oa']:.4f}  "
          f"macro_f1:: {r['macro_f1']:.4f}")
if df3b is not None:
    for _, r in df3b.iterrows():
        print(f"typo  {r['fraksi_typo']:>4.0%}  oa:: {r['oa']:.4f}  "
              f"macro_f1:: {r['macro_f1']:.4f}")
else:
    print("injeksi typo: TIDAK DIKERJAKAN (cc.en.300.bin tidak tersedia)")

print(f"\nseed:: {SEED} (satu seed saja — keterbatasan tahap ini)")